### Data Cleaning

数据处理

---

`Tokenization` + `Embedding` + `POS Encoding`

- `Tokenization`

    分词方法可以以单词为分词，和常见词的词组为分词两种方式

    每个词都有一个id

- `Embedding`

    词的id转为词向量

    1. 可使用**第三方静态表**词嵌入矩阵，id对应词嵌入矩阵的对应行

        *e.g. word2vec(Google, 2013); GloVe(Stanford, 2014); FastText(Facebook, 2016)*

        **特点**：词向量特征稳定，速度较快

    2. 可使用**第三方动态生成**，用一个**神经网络**直接生成 $\mathbf{\hat{Y}}$ 并使用其中间隐藏层 $\mathbf{H}$

        *e.g. BERT(Google, 2018); GPT(OpenAI, 2018); T5(Google, 2020)*

        **特点**：词向量特征更丰富，但速度更慢

    3. OneHot 法，大样本下也趋于完整

- `Position Encoding`

    1. 加上一个**位置矩阵**

        $ p_i^{<t>} = \frac{行}{总行数^{\frac{列}{总列数}}}  \in [0, \infty] $ 实现位置嵌入

        ⚠️ **注意**：行数列数与 `Embedding`**矩阵** 相同，一般行为句长，列为特征维度数
        
        Transformer 最终在此基础上使用三角函数进行压缩

        $ p_i^{<t>} = \begin{cases} 
                        sin(\frac{行}{总行数^{\frac{列}{总列数}}}), 列为偶数 \\
                        cos(\frac{行}{总行数^{\frac{列-1}{总列数}}}), 列为奇数
                      \end{cases} $
        
        这样进行Query, Key值计算后得到的值，可以得到两个元素之间距离的余弦夹角 $ cos(p_i^{<t_1>}-p_i^{<t_2>}) $

### Sequence Labeling

序列标注任务

---


- `Self Attention`

    对于一个 $\mathbf{x}$ 提取出三种不同的矩阵 $\mathbf{V}$, $\mathbf{Q}$, $\mathbf{K}$（三个需训练的权重矩阵 $\mathbf{W}_v, \mathbf{W}_v, \mathbf{W}_v$）

    注意力权重矩阵 $\mathbf{W}_{score} = \mathbf{Q} \mathbf{K}_T$

    中间隐藏层 $\mathbf{H} = \sigma \cdot (\mathbf{W}_{score} \mathbf{V} + \mathbf{B}_x)$ （一个需训练的偏移矩阵 $\mathbf{B}_x$）

    最终预测值 $\hat{\mathbf{Y}} = \sigma \cdot (\mathbf{W}_{y} \mathbf{H} + \mathbf{B}_y)$ （一个系数矩阵 $\mathbf{W}_y$ 与一个需训练的偏移矩阵 $\mathbf{B}_y$）

#### Sequence to One

序列到单值任务

---

对原始序列使用 `Self Attention`, 用 $h^{<1>}$ 或 $h^{<n>}$ 或 $ \frac{h^{<1>} + ... + h^{<n>}}{n} $ 作为特征值来作为这个单值任务的代表特征值。

#### One to Sequence

单值到序列任务

---

用第1个时间步的值预测第2个时间步，再用1, 2时间步预测3时间步（序列到单值任务），以此类推，将单值到序列任务转换为n个序列到单值任务

⚠️ **注意**：只需要计算每个时间步结束后的 $q^{<t>}$ 值即可，通过将 $q^{<t>}\mathbf{K}^T$ 进行键值查询去计算每个时间步的 $\mathbf{V}$, 再从 $\mathbf{V}$ 中取出我们需要的本次时间步的值 $v^{<t>}$

- `Mask Self Attention`

    每个时间步得到的**注意力权重矩阵**，可构成一个下三角的**掩码矩阵**，直接×全量的 $\mathbf{V}$ 矩阵即可直接得到对序列的全预测的隐藏层矩阵。
    $$
    \begin{bmatrix}
    s^{<1>}_{1} \\
    s^{<2>}_{1} & s^{<2>}_{2} \\
    s^{<3>}_{1} & s^{<3>}_{2} & s^{<3>}_{3} \\
    s^{<4>}_{1} & s^{<4>}_{2} & s^{<4>}_{3} & s^{<4>}_{4}
    \end{bmatrix} \cdot \mathbf{V} = \mathbf{H} \rightarrow \mathbf{\hat{Y}}
    $$

    对上三角的掩码就保证了对中间隐藏层进行计算时，每一个时间步的结果都仅仅利用了在此之前的**最后一个时间步的query**与**此时间步所能出现的所有值的Key值**来进行匹配

    注意力权重矩阵 $\mathbf{W}_{score} = \mathbf{Q} \mathbf{K}_T$

    掩码矩阵 $\mathbf{W}_{mask} = mask(\mathbf{W}_{score})$

    中间隐藏层 $\mathbf{H} = \sigma \cdot (\mathbf{W}_{mask} \mathbf{V} + \mathbf{B}_x)$ （一个需训练的偏移矩阵 $\mathbf{B}_x$）

    最终预测值 $\hat{\mathbf{Y}} = \sigma \cdot (\mathbf{W}_{y} \mathbf{H} + \mathbf{B}_y)$ （一个系数矩阵 $\mathbf{W}_y$ 与一个需训练的偏移矩阵 $\mathbf{B}_y$）

#### **Transformer** (Sequence to Sequence)

序列到序列任务

---

**Encoder-Decoder 架构**

- `Cross Attention`

    **原始序列**使用 `Self Attention`, 得到 $\mathbf{V}_{encoder}$, $\mathbf{K}_{encoder}$ 用于对**生成序列**的预测。

    **生成序列** 使用 `Mask Self Attention` 得到 $\mathbf{Q}_{decoder}$

    用 $\mathbf{Q}_{decoder} \mathbf{K}_{encoder}^T$ 查询键值匹配得到合理的**生成序列当前步**的权重矩阵 $\mathbf{W}_{score}$

    中间隐藏层 $\mathbf{H} = \sigma \cdot (\mathbf{W}_{score} \mathbf{V}_{encoder} + \mathbf{B}_x) \rightarrow \hat{\mathbf{Y}}$

- `Multi Head Attention`

    将 $\mathbf{V}, \mathbf{Q}, \mathbf{K}$ 中的 (len, 特征维度) 变成 (head, len, $ \frac{特征维度}{head} $ ) 的三维张量

    e.g. $$ input (5, 512) \rightarrow \begin{cases}
            \mathbf{V} (5, 768) \rightarrow \mathbf{V} (8, 5, 96) \\
            \mathbf{Q} (5, 768) \rightarrow \mathbf{Q} (8, 5, 96) \\
            \mathbf{K} (5, 768) \rightarrow \mathbf{K} (8, 5, 96) \\
            \end{cases} \\
            \rightarrow \mathbf{H} (8, 5, 96) \rightarrow \mathbf{H} (5, 768) $$
    
    为何要使用 **多头注意力机制** ？将特征矩阵拆分成多通道子矩阵可避免提取特征值过程中的数据干扰

- `Attention Block`

    使用多层 `Multi Head Attention Layer` 叠加

    层与层之间使用 **全连接层（FNN）**
    
    进行更充分的特征提取

    ⚠️ **注意**：使用残差链接 ResNet + 层归一化 Layer Normerlization

    `层归一化 Layer Normerlization`: 计算一层 ResNet 均值和标准差后转换为正态分布空间里，防止梯度爆炸。
    
    `Batch Normerlization`: 使用相同位置不同时间步下进行归一化，在此处使用会干扰其他位置的特征值。

- `Softmax`
    